# 🌿 Project 16 — Create an NDVI Time Series

Excellent! This project introduces one of the most important concepts in Remote Sensing:

**Time Series Analysis.**

Until now, you've analyzed **single satellite images** or compared **two dates**.

Now you'll analyze **an entire year** of satellite images and visualize how vegetation changes over time.

This type of analysis is widely used in agriculture, climate science, ecology, and environmental monitoring.

---

# 🎯 Goal

Calculate **NDVI** for many Sentinel-2 images over one year and display the results as a **time series chart**.

Instead of producing another map, you'll create a graph showing how vegetation changes throughout the year.

---

# 🌍 Why is this useful in Remote Sensing?

Time series analysis is used for:

- 🌾 Crop growth monitoring
- 🌿 Seasonal vegetation analysis
- 🌍 Climate change studies
- 🌱 Drought monitoring
- 🔥 Wildfire recovery
- 🌳 Forest health monitoring

Instead of comparing only two dates, we observe vegetation continuously over time.

---

# ⭐ New Skill: Creating Charts

Until now, your workflow looked like this:

```text
Satellite Image
      ↓
Calculate NDVI
      ↓
Display Map
```

Today it becomes:

```text
Image Collection
       ↓
Calculate NDVI
       ↓
One NDVI Image for Each Date
       ↓
Average NDVI
       ↓
Chart
```

Notice something important.

Our final result is **not another map**.

It is a **graph** showing vegetation changes through time.

---

# Before We Start

Previously you worked with:

```text
One Image
```

Now we'll work with:

```text
Many Images
```

Each image has:

- A different acquisition date
- A different NDVI value

Earth Engine will calculate NDVI for **every image automatically**.

---

# Step 1 — Define the Study Area

We'll use Germany.

```javascript
var countryArea = ee.Geometry.Polygon([
  [
    [5.87,47.27],
    [15.04,47.27],
    [15.04,55.06],
    [5.87,55.06],
    [5.87,47.27]
  ]
]);
```

---

# Step 2 — Load the Sentinel-2 Collection

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(countryArea)
.filterDate('2024-01-01','2024-12-31');
```

Notice something.

We do **not** use:

```javascript
.first()
```

Why?

Because we want **all images**, not just one.

---

# Step 3 — Create an NDVI Function

This should look familiar.

```javascript
function calculateNDVI(image){
  return image
    .normalizedDifference(['B8','B4'])
    .rename('NDVI');
}
```

Nothing new here.

The new part is how we'll use this function.

---

# Step 4 — Apply the Function to Every Image ⭐

```javascript
var ndviCollection = collection.map(calculateNDVI);
```

This is today's most important concept.

## What is `.map()`?

Think of it like this:

Before:

```text
Image Collection

Image 1

Image 2

Image 3

Image 4
```

After:

```text
Image 1
   ↓
NDVI

Image 2
   ↓
NDVI

Image 3
   ↓
NDVI

Image 4
   ↓
NDVI
```

`.map()` automatically runs the same function on **every image** in the collection.

The output is still an:

```text
ee.ImageCollection
```

But now every image contains an **NDVI** band.

---

# Step 5 — Create the Time Series Chart ⭐

```javascript
var chart = ui.Chart.image.series({
  imageCollection: ndviCollection,
  region: countryArea,
  reducer: ee.Reducer.mean(),
  scale:10,
  xProperty:'system:time_start'
});
```

This is another major new concept.

Let's understand each part.

## imageCollection

```javascript
imageCollection: ndviCollection
```

The collection we want to analyze.

---

## region

```javascript
region: countryArea
```

The area where NDVI will be measured.

---

## reducer

```javascript
ee.Reducer.mean()
```

Each image contains **millions of pixels**.

We can't plot millions of values for one date.

Instead, Earth Engine calculates:

> **The average NDVI of the study area.**

So each date has **one value**.

---

## scale

```javascript
scale:10
```

Sentinel-2 spatial resolution.

---

## xProperty

```javascript
system:time_start
```

This tells Earth Engine:

> Use the image acquisition date on the x-axis.

---

# Step 6 — Print the Chart

```javascript
print(chart);
```

The chart appears in the **Console**.

Example:

```text
NDVI
↑
│          ▲
│      ▲
│   ▲
│ ▲
└──────────────────→ Time
```

Each point represents the **average NDVI** for one satellite image.

---

# ⭐ Best Practice — Monthly NDVI Composites

In practice, satellite images are not collected at perfectly regular intervals.

Some images may also contain clouds.

Instead of plotting every image, many researchers create **monthly NDVI composites**.

Advantages:

- Fewer noisy observations
- Reduced cloud effects
- Easier seasonal interpretation
- Standard approach in scientific studies

---

## Step 1 — Filter Cloudy Images

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(countryArea)
.filterDate('2024-01-01', '2024-12-31')
.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',20));
```

Only images with less than **20% cloud cover** are used.

---

## Step 2 — Add NDVI as a New Band

Instead of replacing the original image, we add NDVI as an additional band.

```javascript
function addNDVI(image){
  return image.addBands(
    image.normalizedDifference(['B8','B4'])
      .rename('NDVI')
  );
}

var ndviCollection = collection.map(addNDVI);
```

Now every image contains:

```text
B2
B3
B4
B8
...
NDVI
```

---

## Step 3 — Create Monthly Composites ⭐

```javascript
var months = ee.List.sequence(1,12);

var monthlyNDVI = ee.ImageCollection.fromImages(
  months.map(function(month){

    var monthlyImage = ndviCollection
      .filter(
        ee.Filter.calendarRange(
          month,
          month,
          'month'
        )
      )
      .select('NDVI')
      .median();

    return monthlyImage.set(
      'system:time_start',
      ee.Date.fromYMD(
        2024,
        month,
        1
      ).millis()
    );

  })
);
```

### What's happening?

For every month:

```text
January
↓

All January Images
↓

Median NDVI
↓

One Monthly Image
```

Repeat for:

```text
February
↓

March
↓

...

↓

December
```

Result:

```text
12 NDVI Images

↓

One Image Per Month
```

---

## Step 4 — Create the Monthly Chart

```javascript
var chart = ui.Chart.image.series({
  imageCollection: monthlyNDVI,
  region: countryArea,
  reducer: ee.Reducer.mean(),
  scale:500,
  xProperty:'system:time_start'
})
.setOptions({
  title:'Monthly Mean NDVI of Germany (2024)',
  hAxis:{
    title:'Month'
  },
  vAxis:{
    title:'NDVI'
  },
  lineWidth:2,
  pointSize:5
});
```

Now the chart contains **12 points** instead of dozens of irregular observations.

---

## Step 5 — Print the Chart

```javascript
print(chart);
```

---

# Complete Code — Basic Time Series

```javascript
// 1. Study Area
var countryArea = ee.Geometry.Polygon([
  [
    [5.87,47.27],
    [15.04,47.27],
    [15.04,55.06],
    [5.87,55.06],
    [5.87,47.27]
  ]
]);

// 2. Load Collection
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(countryArea)
.filterDate('2024-01-01','2024-12-31');

// 3. NDVI Function
function calculateNDVI(image){
  return image
    .normalizedDifference(['B8','B4'])
    .rename('NDVI');
}

// 4. Apply to Every Image
var ndviCollection = collection.map(calculateNDVI);

// 5. Create Chart
var chart = ui.Chart.image.series({
  imageCollection: ndviCollection,
  region: countryArea,
  reducer: ee.Reducer.mean(),
  scale:10,
  xProperty:'system:time_start'
});

// 6. Print
print(chart);
```

---

# Complete Code — Monthly NDVI Composites (Best Practice)

```javascript
// =====================================
// 1. Study Area (Germany)
// =====================================
var countryArea = ee.Geometry.Polygon([
  [
    [5.87, 47.27],
    [15.04, 47.27],
    [15.04, 55.06],
    [5.87, 55.06],
    [5.87, 47.27]
  ]
]);

// =====================================
// 2. Load Sentinel-2 Collection
// =====================================
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(countryArea)
.filterDate('2024-01-01','2024-12-31')
.filter(
  ee.Filter.lt(
    'CLOUDY_PIXEL_PERCENTAGE',
    20
  )
);

// =====================================
// 3. Function to Add NDVI
// =====================================
function addNDVI(image){
  return image.addBands(
    image.normalizedDifference(['B8','B4'])
      .rename('NDVI')
  );
}

var ndviCollection = collection.map(addNDVI);

// =====================================
// 4. Create Monthly NDVI Composites
// =====================================
var months = ee.List.sequence(1,12);

var monthlyNDVI =
ee.ImageCollection.fromImages(

  months.map(function(month){

    var monthlyImage =
      ndviCollection
      .filter(
        ee.Filter.calendarRange(
          month,
          month,
          'month'
        )
      )
      .select('NDVI')
      .median();

    return monthlyImage.set(
      'system:time_start',
      ee.Date.fromYMD(
        2024,
        month,
        1
      ).millis()
    );

  })

);

// =====================================
// 5. Create Time Series Chart
// =====================================
var chart = ui.Chart.image.series({
  imageCollection: monthlyNDVI,
  region: countryArea,
  reducer: ee.Reducer.mean(),
  scale:500,
  xProperty:'system:time_start'
})
.setOptions({
  title:'Monthly Mean NDVI of Germany (2024)',
  hAxis:{
    title:'Month'
  },
  vAxis:{
    title:'NDVI'
  },
  lineWidth:2,
  pointSize:5
});

// =====================================
// 6. Print Chart
// =====================================
print(chart);
```

---

# Things to Remember

## 1. `.map()`

One of the most important Earth Engine functions.

```text
Image Collection
       ↓
.map()
       ↓
Processed Image Collection
```

It applies the same operation to every image.

---

## 2. `ui.Chart.image.series()`

Creates a time series chart from an image collection.

```text
X-axis → Time

Y-axis → Value calculated from each image
```

---

## 3. `ee.Reducer.mean()`

Instead of plotting millions of pixels,

Earth Engine calculates the **average NDVI** for the selected region.

---

## 4. Monthly Composites

Creating one representative image per month:

- Reduces cloud contamination
- Smooths noisy observations
- Makes seasonal trends easier to interpret
- Is widely used in scientific research

---

## 5. `calendarRange()`

```javascript
ee.Filter.calendarRange(month, month, 'month')
```

Filters images by calendar month.

Example:

```text
Month = 7

↓

July Images Only
```

---

## 6. `median()`

Creates a representative monthly image by taking the **median** value for each pixel.

Compared with a single image, the median:

- Reduces cloud effects
- Reduces outliers
- Produces a more stable monthly composite

---

# Common Beginner Mistakes

❌ **Mistake 1:** Using `.first()`.

```javascript
collection.first()
```

This returns only one image, so you cannot create a time series.

---

❌ **Mistake 2:** Forgetting `.map()`.

Without `.map()`, the collection doesn't contain an NDVI band, so the chart cannot plot NDVI.

---

❌ **Mistake 3:** Confusing a map with a chart.

A **map** shows **where** values occur.

A **chart** shows **how** values change over time.

---

❌ **Mistake 4:** Using every image without filtering clouds.

Clouds can lower NDVI values and create noisy charts.

Filtering cloudy images or creating monthly composites produces more reliable results.

---

# Your Practice (Project 16.1)

Create an NDVI time series for a study area of your choice.

## Requirements

1. Create a study area.
2. Load the Sentinel-2 collection for one year.
3. Filter cloudy images.
4. Create an `addNDVI()` function.
5. Apply it using `.map()`.
6. Create monthly NDVI composites.
7. Create an NDVI time series chart.
8. Print the chart.

---

Before writing the code, answer these two questions.

## Question 1

Why do we use:

```javascript
collection.map(addNDVI)
```

instead of:

```javascript
addNDVI(collection.first())
```

when creating a time series?

---

## Question 2

Why do we use:

```javascript
ee.Reducer.mean()
```

when creating the chart instead of plotting every pixel value?

Send me your answers and your code, and I'll review both your understanding and your implementation exactly like a university professor.